# NLLB-600M: полный прогон 100k (EN→RU + RU→EN)

**Предусловие:** запусти `benchmark_t4.ipynb` и убедись, что скорость ≥7 req/s.

**Что делает:**
1. Переводит 100k запросов EN→RU
2. Переводит 100k запросов RU→EN
3. Кеш сохраняется в Google Drive каждые 1500 запросов

**Время:** ~3–4 часа (если бенчмарк показал ≥7 req/s)

## 1. Проверка GPU

In [ ]:
!nvidia-smi

## 2. Монтируем Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Клонируем репозиторий (если ещё не клонирован)

In [ ]:
import os
if not os.path.exists('/content/kry_prompt-lib'):
    !git clone https://github.com/Krynovv/kry_prompt-lib.git
%cd /content/kry_prompt-lib/kursovaya-tex

## 4. Устанавливаем зависимости

In [ ]:
!pip install -q torch sentencepiece sacremoses transformers

## 5. Настраиваем окружение

In [ ]:
import os
import sys

# Путь кеша в Google Drive (переживёт разрыв сессии)
os.environ['SEMSEARCH_CACHE_DIR'] = '/content/drive/MyDrive/semsearch_cache'

# Добавляем текущую директорию в PYTHONPATH для импорта модулей
sys.path.insert(0, '.')

print(f"Кеш будет сохраняться в: {os.environ["SEMSEARCH_CACHE_DIR"]}")
print(f"Рабочая директория: {os.getcwd()}")

## 6. Перевод EN→RU (100k запросов)

In [ ]:
import time
start = time.time()

!python -m code.semsearch.data.translate \
  --model nllb-en2ru \
  --input code/semsearch/data/train_queries.txt \
  --output /content/drive/MyDrive/semsearch_cache/train_queries_ru.txt \
  --device cuda \
  --dtype float16 \
  --flush-every 1500

en2ru_time = (time.time() - start) / 3600
print(f"\n✅ EN→RU завершён за {en2ru_time:.2f} часов")

## 7. Перевод RU→EN (100k запросов)

In [ ]:
start = time.time()

!python -m code.semsearch.data.translate \
  --model nllb-ru2en \
  --input /content/drive/MyDrive/semsearch_cache/train_queries_ru.txt \
  --output /content/drive/MyDrive/semsearch_cache/train_queries_ru2en.txt \
  --device cuda \
  --dtype float16 \
  --flush-every 1500

ru2en_time = (time.time() - start) / 3600
print(f"\n✅ RU→EN завершён за {ru2en_time:.2f} часов")

## 8. Итоговая статистика

In [ ]:
total_time = en2ru_time + ru2en_time
print(f"\n{'='*60}")
print(f"ПОЛНЫЙ ПРОГОН ЗАВЕРШЁН")
print(f"{'='*60}")
print(f"EN→RU: {en2ru_time:.2f} часов")
print(f"RU→EN: {ru2en_time:.2f} часов")
print(f"Всего: {total_time:.2f} часов")
print(f"{'='*60}")
print(f"\nРезультаты в Google Drive:")
print(f"- train_queries_ru.txt (EN→RU)")
print(f"- train_queries_ru2en.txt (RU→EN)")
print(f"\nКеш: /content/drive/MyDrive/semsearch_cache/translation/")

## Что дальше?

1. **Скопируй результаты** из Google Drive:
   - `train_queries_ru.txt` → `kursovaya-tex/code/semsearch/data/`
   - `train_queries_ru2en.txt` → `kursovaya-tex/code/semsearch/data/`

2. **Запиши в research-log.md:**
   ```
   ## 2026-11-21: Полный перевод 100k
   
   - EN→RU: X.XX часов
   - RU→EN: X.XX часов
   - Модель: NLLB-600M, fp16, T4, beams=4, batch=16
   ```